# C4 — Notebook 05: Static environmental features (one catchment → all 40)

**Goal:** Independently extract the same **nine environmental descriptors** from each of the 40 **existing candidate upstream polygons**. This is the C4 *breadth* representation: no trained C1–C3 features and no additional HAND/TWI processing.

**Pipeline stages:**
1. Validate the 40 local catchments and the target-free Notebook 01 station registry.
2. **Pilot:** inspect **Holombuwa** (a relatively small Kelani catchment) with real Earth Engine and local HydroRIVERS data. Check all nine values, valid coverage and original-polygon area.
3. **Bulk:** only after pilot approval, process all 40. Save **one static vector per station**, *not* 1,429 Earth Engine extractions. The later C4 loader joins that vector onto each station-day by `station_id`.
4. Cache source-derived summaries by station and a source/configuration fingerprint; reruns use validated local cache without re-requesting Earth Engine.

**Real sources:** `COPERNICUS/DEM/GLO30` (DSM elevation + slope), `ESA/WorldCover/v200` (2021 classes), `JRC/GSW1_4/GlobalSurfaceWater` (1984–2021 historical transition), and a **locally available licensed HydroRIVERS Asia** vector dataset. HydroRIVERS is a different network from the MERIT flow raster that formed the polygons.

**Research limitations:** Results are modelled catchment attributes, not surveyed river lengths, true basin boundaries, or observed 2025 flood extent. Input areas retain existing spatial-QA flags. No training labels are read in this notebook.

In [ ]:
# Cell 1 — Imports and repository discovery 
from pathlib import Path
import hashlib, json, math, os, re, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import shape, mapping
from shapely.ops import transform as shapely_transform
from pyproj import Transformer

REPO_ROOT_OVERRIDE = None  

def find_repo_root():
    if REPO_ROOT_OVERRIDE is not None:
        p = Path(REPO_ROOT_OVERRIDE).resolve()
        if (p/'c4_fusion').is_dir() and (p/'shared'/'spatial'/'catchments').is_dir():
            return p
        raise FileNotFoundError(f'Invalid project root override: {p}')
    for base in (Path.cwd().resolve(),):
        for p in (base, *base.parents):
            if (p/'c4_fusion').is_dir() and (p/'shared'/'spatial'/'catchments').is_dir():
                return p
    raise FileNotFoundError('Cannot find J26-DS-314 repo root. Set REPO_ROOT_OVERRIDE above.')

ROOT = find_repo_root()
INDEX_FILE = ROOT/'data/processed/c4/manifests/c4_sample_index_1429.csv'
REGISTRY_FILE = ROOT/'data/processed/c4/manifests/c4_station_registry_40_no_coordinates.csv'
POLYGON_ROOT = ROOT/'shared/spatial/catchments'
OUT_DIR = ROOT/'data/processed/c4/environment'
CACHE_DIR = OUT_DIR/'cache'
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# All named constants below are fixed for this PP1 extraction version.
DEM_ASSET = 'COPERNICUS/DEM/GLO30'
WORLDCOVER_ASSET = 'ESA/WorldCover/v200'
JRC_ASSET = 'JRC/GSW1_4/GlobalSurfaceWater'
ANALYSIS_CRS = 'EPSG:32644'  # UTM 44N, only for metric measurement/reduction
DEM_SCALE_M, WORLDCOVER_SCALE_M, JRC_SCALE_M = 30, 10, 30
PILOT_STATION_ID = 'holombuwa'
EE_PROJECT = os.getenv('GOOGLE_EARTH_ENGINE_PROJECT','').strip() or None

# HYDRORIVERS MUST BE AN ACTUAL VECTOR DATASET. No MERIT or OSM substitute.
HYDRORIVERS_PATH = ROOT/'c4_fusion/private_data/hydrorivers/HydroRIVERS_v10_as.shp'

FEATURE_COLS = [
    'env_log1p_area_km2',
    'env_mean_elevation_m',
    'env_mean_slope_deg',
    'env_tree_fraction',
    'env_cropland_fraction',
    'env_built_fraction',
    'env_worldcover_water_fraction',
    'env_jrc_persistent_water_fraction',
    'env_hydrorivers_density_km_per_km2',
]
print('REPO:', ROOT)
print('INPUT index exists:', INDEX_FILE.is_file())
print('INPUT 40-station registry exists:', REGISTRY_FILE.is_file())
print('HYDRORIVERS local file exists:', HYDRORIVERS_PATH.is_file())

In [6]:
# Cell 2 — Read Notebook 01 outputs and verify exactly 40 source polygons
assert INDEX_FILE.is_file(), 'Run Notebook 01 first: missing c4_sample_index_1429.csv'
assert REGISTRY_FILE.is_file(), 'Run Notebook 01 first: missing c4_station_registry_40_no_coordinates.csv'
index = pd.read_csv(INDEX_FILE, dtype={'station_id':str}, low_memory=False)
registry = pd.read_csv(REGISTRY_FILE, dtype={'station_id':str}, low_memory=False)
assert len(index) == 1429 and index['record_id'].is_unique
assert len(registry) == 40 and registry['station_id'].is_unique
assert index['station_id'].nunique() == 39
assert index['station_id'].isin(registry['station_id']).all()
assert PILOT_STATION_ID in set(registry['station_id']), 'Pilot station not in 40-station registry'

def slug(s):
    return re.sub(r'_+', '_', re.sub(r'[^a-z0-9]+', '_', str(s).casefold())).strip('_')

def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

def canonical_path(relative):
    relative=str(relative).replace('\\','/')
    p=(ROOT/relative).resolve()
    if not p.is_relative_to(ROOT.resolve()):
        raise ValueError(f'Path outside repository: {relative}')
    if not relative.startswith('shared/spatial/catchments/'):
        raise ValueError(f'Unexpected catchment directory: {relative}')
    return p

paths={}
for sid, group in index.groupby('station_id', sort=True):
    candidates=group['polygon_repo_path'].dropna().unique()
    assert len(candidates)==1, f'Multiple or missing polygon references for {sid}'
    paths[sid]=canonical_path(candidates[0])
# The 40th station has a candidate polygon but no labels.
paths['thambuththegama'] = POLYGON_ROOT/'kala_oya/polygons/thambuththegama_catchment_polygon_CANDIDATE.geojson'
assert set(paths)==set(registry['station_id']), '40-polygon registry mismatch; stop instead of guessing paths.'

# The original irregular boundaries are never modified.
polygons={}
for sid, path in sorted(paths.items()):
    assert path.is_file(), f'Catchment source is missing: {path}'
    doc=json.loads(path.read_text(encoding='utf-8'))
    features=doc.get('features') if doc.get('type')=='FeatureCollection' else [doc]
    assert len(features)==1, f'Expected one candidate polygon for {sid}'
    feat=features[0]
    claimed=feat.get('properties',{}).get('station') or feat.get('properties',{}).get('station_name')
    assert claimed is None or slug(claimed)==sid, f'Catchment identity does not match {sid}'
    geom=shape(feat['geometry'])
    assert geom.geom_type in ('Polygon','MultiPolygon') and geom.is_valid and not geom.is_empty
    polygons[sid]=geom
print(f'PASS: {len(polygons)} distinct existing candidate polygons loaded (EPSG:4326).')
print('No future target columns loaded; notebook operates once per catchment.')

PASS: 40 distinct existing candidate polygons loaded (EPSG:4326).
No future target columns loaded; notebook operates once per catchment.


In [7]:
# Cell 3 — Check HydroRIVERS local source and initialize Earth Engine
if not HYDRORIVERS_PATH.is_file():
    raise FileNotFoundError(
        'HydroRIVERS Asia shapefile is missing. Put HydroRIVERS_v10_as.shp '
        '(with .dbf/.shx/.prj) at the configured private path. '
        'Do NOT invent river density or substitute MERIT river cells silently.'
    )
for suffix in ('.shx','.dbf','.prj'):
    assert HYDRORIVERS_PATH.with_suffix(suffix).is_file(), f'Missing HydroRIVERS shapefile sidecar: {suffix}'
try:
    import ee
except ImportError as exc:
    raise ImportError('Install earthengine-api into your selected VS Code Python .venv.') from exc
try:
    if EE_PROJECT:
        ee.Initialize(project=EE_PROJECT)
    else:
        ee.Initialize()  
except Exception as exc:
    raise RuntimeError('Earth Engine initialization failed: run your one-time GEE authorization/configuration in your own environment; then rerun this cell.') from exc
print('Earth Engine initialized. This cell downloads no raster imagery.')

Earth Engine initialized. This cell downloads no raster imagery.


In [8]:
# Cell 4 — Local helper functions: exact polygon area and local HydroRIVERS length
# Importantly, hydrography lengths are calculated in metres, NOT latitude/longitude degrees.
from shapely.geometry import box
TRANSFORM_UTM = Transformer.from_crs('EPSG:4326', ANALYSIS_CRS, always_xy=True)

def projected_geometry(geometry):
    return shapely_transform(TRANSFORM_UTM.transform, geometry)

def local_catchment_area_km2(geom):
    return projected_geometry(geom).area / 1_000_000.0

# Read only Sri Lanka's area, not the whole continent, from the local HydroRIVERS vector.
rivers=gpd.read_file(HYDRORIVERS_PATH, bbox=(79.0,5.5,82.5,10.0), engine='pyogrio')
assert rivers.crs is not None, 'HydroRIVERS CRS missing; do not measure lengths in unknown units.'
rivers=rivers.to_crs('EPSG:4326')
rivers=rivers[rivers.geometry.notna() & ~rivers.geometry.is_empty].copy()
assert len(rivers)>0, 'No HydroRIVERS features in Sri Lanka. Check that you have the Asia file.'
assert rivers.geom_type.isin(['LineString','MultiLineString']).all(), 'Unexpected HydroRIVERS geometry types.'
print('Loaded local HydroRIVERS features inside Sri Lanka vicinity:',len(rivers))

# Spatial index makes repeat per-catchment line clipping efficient.
RIVER_UTM = rivers.to_crs(ANALYSIS_CRS)
RIVER_SINDEX = RIVER_UTM.sindex

def river_density(geom, basin_area_km2):
    basin=projected_geometry(geom)
    matches=list(RIVER_SINDEX.query(basin, predicate='intersects'))
    if not matches:
        return {'hydro_length_km':0.0,'hydro_count_intersecting':0,
                'env_hydrorivers_density_km_per_km2':0.0,
                'hydrorivers_note':'No mapped HydroRIVERS reach intersects the catchment (not proof of no small streams).'}
    clipped = RIVER_UTM.geometry.iloc[matches].intersection(basin)
    total_km=float(clipped.length.sum()/1000)
    return {'hydro_length_km':total_km,'hydro_count_intersecting':len(matches),
            'env_hydrorivers_density_km_per_km2':total_km/basin_area_km2,
            'hydrorivers_note':'Modelled HydroRIVERS network, clipped to original candidate polygon.'}

Loaded local HydroRIVERS features inside Sri Lanka vicinity: 5723


In [12]:
# Cell 5 — Real GEE area-weighted polygon statistics, with native-resolution reductions
# WorldCover is categorical: never take the numeric mean of its class codes.
# JRC uses documented classes 1,2,7; seasonality==12 is QA-only.

def gee_polygon(geom):
    # GeoJSON coordinates are WGS84 lon/lat; GEE handles CRS conversion internally.
    return ee.Geometry(mapping(geom), 'EPSG:4326', geodesic=False)

def ee_area_stats(image, region, native_scale):
    # Each band is already 'pixel area * quantity' or valid pixel area.
    return image.reduceRegion(
        reducer=ee.Reducer.sum(), geometry=region,
        crs=ANALYSIS_CRS, scale=native_scale,
        maxPixels=1_000_000_000, tileScale=8
    ).getInfo()

def required_number(stats, name):
    v=stats.get(name)
    if v is None or not np.isfinite(float(v)):
        raise ValueError(f'GEE returned missing/nonfinite {name}; check geographic source coverage.')
    return float(v)

def gee_dem_metrics(region):
    col=ee.ImageCollection(DEM_ASSET).filterBounds(region).select('DEM')
    # Copernicus GLO30 is an ImageCollection. Explicitly restore native projection
    # after mosaicking so Terrain.slope is not computed on a 1-degree default grid.
    dem=col.mosaic().setDefaultProjection(col.first().projection()).rename('dem')
    slope=ee.Terrain.slope(dem).rename('slope')
    area=ee.Image.pixelArea()
    image=ee.Image.cat([
        dem.multiply(area).rename('elev_num'),
        area.updateMask(dem.mask()).rename('elev_den'),
        slope.multiply(area).rename('slope_num'),
        area.updateMask(slope.mask()).rename('slope_den'),
    ])
    result=ee_area_stats(image, region, DEM_SCALE_M)
    ed=required_number(result,'elev_den'); sd=required_number(result,'slope_den')
    assert ed>0 and sd>0, 'Copernicus DEM has no valid area'
    return {'env_mean_elevation_m':required_number(result,'elev_num')/ed,
            'env_mean_slope_deg':required_number(result,'slope_num')/sd,
            'dem_valid_area_km2':ed/1e6, 'slope_valid_area_km2':sd/1e6}

def gee_worldcover_metrics(region):
    cover=ee.ImageCollection(WORLDCOVER_ASSET).first().select('Map')
    area=ee.Image.pixelArea()
    image=ee.Image.cat([
        area.updateMask(cover.mask()).rename('wc_den'),
        *[area.multiply(cover.eq(value)).rename(name) for value,name in [
            (10,'tree_num'),(40,'crop_num'),(50,'built_num'),(80,'water_num')]]
    ])
    result=ee_area_stats(image, region, WORLDCOVER_SCALE_M)
    den=required_number(result,'wc_den')
    assert den>0, 'ESA WorldCover contains no valid classified pixels'
    return {'env_tree_fraction':required_number(result,'tree_num')/den,
            'env_cropland_fraction':required_number(result,'crop_num')/den,
            'env_built_fraction':required_number(result,'built_num')/den,
            'env_worldcover_water_fraction':required_number(result,'water_num')/den,
            'worldcover_valid_area_km2':den/1e6}

def gee_jrc_metrics(region):
    # JRC masks areas where water has never been detected. For a catchment-wide
    # fraction, those locations must NOT disappear from the denominator.
    # Treat them as "no mapped persistent water" (zero), NOT as JRC observations.
    jrc = ee.Image(JRC_ASSET)
    transition = jrc.select('transition')
    seasonality = jrc.select('seasonality')
    area = ee.Image.pixelArea()
    # First calculate categories on the original transition band, THEN
    # unmask to zero so dry/unmapped land has water-indicator value 0.
    persistent = (transition.eq(1).Or(transition.eq(2)).Or(transition.eq(7)))        .unmask(0).rename('persistent_indicator')
    season12 = seasonality.eq(12).unmask(0).rename('season12_indicator')
    any_mapped_water = jrc.select('max_extent').eq(1).unmask(0)
    image = ee.Image.cat([
        area.rename('jrc_area_den'),
        area.multiply(persistent).rename('jrc_water_num'),
        area.multiply(season12).rename('season12_num'),
        area.multiply(any_mapped_water).rename('jrc_any_water_num'),
    ])
    result = ee_area_stats(image, region, JRC_SCALE_M)
    den = required_number(result, 'jrc_area_den')
    if den <= 0:
        raise ValueError('JRC area denominator is zero: inspect catchment geometry and GEE reduction.')
    numerator = required_number(result, 'jrc_water_num')
    season_num = required_number(result, 'season12_num')
    any_water_num = required_number(result, 'jrc_any_water_num')
    fraction = numerator / den
    if not (0 <= fraction <= 1.001):
        raise ValueError(f'Unexpected JRC persistent-water fraction: {fraction}')
    return {
        'env_jrc_persistent_water_fraction': fraction,
        'jrc_catchment_denominator_area_km2': den / 1e6,
        'jrc_seasonality12_fraction_QA_ONLY': season_num / den,
        'jrc_ever_mapped_water_fraction_QA_ONLY': any_water_num / den,
        'jrc_masked_pixels_assumed_nonpersistent': True,
        'jrc_observation_coverage_verified': False,
    }

### Resolution and interpretation QA

- Copernicus DEM elevation and slope are summarized at **30 m** nominal scale; GLO-30 is a **digital surface model**. Both outputs are means over valid classified catchment area.
- ESA WorldCover 2021 classes **10 trees, 40 cropland, 50 built-up, 80 permanent water** are summarized at the source's **10 m** nominal scale. Class fractions are normalized by valid WorldCover area—not by the sum of the four selected classes.
- JRC GSW v1.4 `transition` **1, 2, 7** forms one historical persistent-water fraction. `seasonality == 12` is a separate *diagnostic*, **not** a tenth input.
- For every raster, we preserve the fraction of the polygon covered by valid source pixels. We do **not** replace no-data with zero or silently change reduction resolution through `bestEffort`.
- The vector line density counts only mapped HydroRIVERS reaches intersecting the *original* polygon; small streams may be absent from this global product.

**Large catchments:** WorldCover's 10 m reduction can be expensive. Pilot first, cache successes, and review GEE quotas/timeouts before processing all 40. If a request fails, this notebook records the failure; it does not invent an approximate result or lower resolution silently.

In [13]:
# Cell 6 — Deterministic fingerprint and retry-safe caching, once per station
# For each station: if cached file's fingerprint matches, GEE is NOT called.
def source_hashes_for_hydro():
    return {suffix:sha256_file(HYDRORIVERS_PATH.with_suffix(suffix)) for suffix in ('.shp','.shx','.dbf','.prj')}
HYDRO_HASHES=source_hashes_for_hydro()
SETTINGS={
    'feature_version':'C4_ENV_PILOT_V2_JRC_FULL_POLYGON', 'metric_crs':ANALYSIS_CRS,
    'dem_asset':DEM_ASSET,'dem_scale':DEM_SCALE_M,
    'worldcover_asset':WORLDCOVER_ASSET,'worldcover_scale':WORLDCOVER_SCALE_M,
    'jrc_asset':JRC_ASSET,'jrc_scale':JRC_SCALE_M,
    'jrc_persistent_classes':[1,2,7], 'worldcover_classes':[10,40,50,80],
    'hydrorivers_hashes':HYDRO_HASHES,
}

def cache_fingerprint(station_id):
    record={'station_id':station_id,'polygon_sha256':sha256_file(paths[station_id]),'settings':SETTINGS}
    return hashlib.sha256(json.dumps(record,sort_keys=True).encode()).hexdigest()

def write_json_if_changed(path, data):
    import tempfile
    raw=(json.dumps(data,indent=2,sort_keys=True,allow_nan=False)+'\n').encode('utf-8')
    if path.is_file() and path.read_bytes()==raw: return 'UNCHANGED'
    tmp=path.with_name(path.name+'.writing.tmp')
    tmp.write_bytes(raw)
    os.replace(tmp,path)
    return 'WRITTEN'

def calculate_one_station(station_id, use_cache=True):
    geom=polygons[station_id]
    fingerprint=cache_fingerprint(station_id)
    cache_file=CACHE_DIR/f'{station_id}.json'
    if use_cache and cache_file.is_file():
        try:
            prior=json.loads(cache_file.read_text(encoding='utf-8'))
            if prior.get('fingerprint')==fingerprint and all(np.isfinite(float(prior['features'][x])) for x in FEATURE_COLS):
                return prior, 'CACHED — zero Earth Engine queries'
        except (KeyError,ValueError,TypeError):
            pass
    area_km2=local_catchment_area_km2(geom)
    if area_km2<=0: raise ValueError(f'Nonpositive projected area: {station_id}')
    location=gee_polygon(geom)

    # PARTIAL CHECKPOINT: if a source fails, rerun without repeating successful
    # DEM / WorldCover / JRC requests made earlier in this same station.
    progress_file=CACHE_DIR/f'{station_id}.progress.json'
    progress={'fingerprint':fingerprint,'parts':{}}
    if progress_file.is_file() and use_cache:
        try:
            previous=json.loads(progress_file.read_text(encoding='utf-8'))
            if previous.get('fingerprint') == fingerprint:
                progress = previous
            else:
                # V1 computed JRC's denominator from only unmasked water-related pixels.
                # We can safely reuse successful V1 DEM/WorldCover parts if their
                # polygon SHA and all non-JRC settings are identical to V2.
                old_settings = dict(SETTINGS, feature_version='C4_ENV_PILOT_V1')
                old_record = {
                    'station_id': station_id,
                    'polygon_sha256': sha256_file(paths[station_id]),
                    'settings': old_settings,
                }
                old_fingerprint = hashlib.sha256(
                    json.dumps(old_record, sort_keys=True).encode()
                ).hexdigest()
                if previous.get('fingerprint') == old_fingerprint:
                    reusable = {
                        key: value for key, value in previous.get('parts', {}).items()
                        if key in ('dem', 'worldcover')
                    }
                    progress = {'fingerprint': fingerprint, 'parts': reusable}
                    write_json_if_changed(progress_file, progress)
                    print(f'{station_id}: preserved verified source-specific DEM/WorldCover partial cache after JRC v2 fix.')
        except (ValueError,TypeError):
            pass

    def get_part(name, fn):
        if name in progress['parts']:
            return progress['parts'][name]
        value=fn()
        progress['parts'][name]=value
        write_json_if_changed(progress_file,progress)
        return value

    dem=get_part('dem', lambda:gee_dem_metrics(location))
    wc=get_part('worldcover', lambda:gee_worldcover_metrics(location))
    jrc=get_part('jrc', lambda:gee_jrc_metrics(location))
    hydro=get_part('hydrorivers', lambda:river_density(geom,area_km2))
    record={
        'station_id':station_id,
        'basin':registry.set_index('station_id').loc[station_id,'basin'],
        'polygon_repo_path':str(paths[station_id].relative_to(ROOT)).replace('\\','/'),
        'polygon_sha256':sha256_file(paths[station_id]),
        'fingerprint':fingerprint,
        'features':{'env_log1p_area_km2':float(np.log1p(area_km2)),
                   **{k:v for d in [dem,wc,jrc,hydro] for k,v in d.items() if k in FEATURE_COLS}},
        'qa':{'original_polygon_area_km2_utm':area_km2,
              **{k:v for d in [dem,wc,jrc,hydro] for k,v in d.items() if k not in FEATURE_COLS}},
    }
    assert len(record['features'])==9, f'Missing environment feature in {station_id}: {record["features"]}'
    for name,v in record['features'].items():
        assert np.isfinite(float(v)), f'Nonfinite feature: {station_id} / {name}'
    for name in [x for x in FEATURE_COLS if x.endswith('_fraction')]:
        assert -1e-6<=record['features'][name]<=1+1e-6, f'Invalid fraction in {name}'
    for key in ('dem_valid_area_km2','slope_valid_area_km2','worldcover_valid_area_km2','jrc_catchment_denominator_area_km2'):
        valid_fraction=record['qa'][key]/area_km2
        record['qa'][key.replace('_area_km2','_coverage_fraction')]=valid_fraction
        if not (0.90<=valid_fraction<=1.10):
            raise ValueError(f'{station_id}: {key} area ratio {valid_fraction:.3f} implausible — check raster projection/coverage before saving')
    # JRC denominator-area agreement is NOT historical observation coverage.
    write_json_if_changed(cache_file,record)
    if progress_file.exists():
        progress_file.unlink()  # final cache is authoritative; no duplicate staging file
    return record, 'NEW — complete GEE/local outputs, with partial-result resume support'

## Phase 1 — Pilot on **one real catchment**

We use Holombuwa because its candidate catchment is smaller than the major downstream basins, this shortens our initial 10 m WorldCover query.

In [ ]:
# Cell 7 — Real pilot extraction (only ONE station, with caching)
pilot, pilot_status = calculate_one_station(PILOT_STATION_ID)
print('Pilot:', PILOT_STATION_ID, '|', pilot_status)
pd.set_option('display.max_rows',50)
display(pd.DataFrame([pilot['features']],index=[PILOT_STATION_ID]).T.round(6).rename(columns={PILOT_STATION_ID:'value'}))
print('QA metadata, raster coverage and JRC denominator diagnostics:')
display(pd.Series(pilot['qa'],name='pilot_qa').to_frame())
print('Source polygon SHA256:',pilot['polygon_sha256'])
print('Pilot cache:',CACHE_DIR/f'{PILOT_STATION_ID}.json')

## Phase 2 — All 40 catchments 

**Each station is computed once**; already finished station caches are reused. Failure for one station is logged and other stations continue. Do **not** treat zero water fraction as missing automatically—JRC historical persistent-water can legitimately be 0.

In [15]:
# Cell 8 — Deliberate guard against accidentally launching 40 GEE extractions
RUN_ALL_40 = True
assert 'pilot' in globals(), 'Pilot has not succeeded yet. Run Cell 7 first.'
assert len(pilot['features']) == 9
assert all(np.isfinite(float(pilot['features'][name])) for name in FEATURE_COLS)
print('Pilot passed 9-feature structure and finiteness tests.')
print('Bulk enabled:', RUN_ALL_40)

Pilot passed 9-feature structure and finiteness tests.
Bulk enabled: True


In [ ]:
# Cell 9 — Compute remaining stations, persist partial successes, report failures.
# This does not submit any Earth Engine requests unless RUN_ALL_40=True.
import time
bulk_rows = []
bulk_errors = []
if not RUN_ALL_40:
    print('Bulk is OFF. Change RUN_ALL_40=True in Cell 8 after pilot inspection, then rerun Cells 8–9.')
else:
    for number, sid in enumerate(sorted(polygons), start=1):
        try:
            record, status = calculate_one_station(sid)
            bulk_rows.append(record)
            print(f'{number:02d}/40  {sid}: {status}', flush=True)
        except Exception as exc:
            bulk_errors.append({'station_id':sid, 'exception':type(exc).__name__, 'message':str(exc)})
            print(f'{number:02d}/40  {sid}: FAILED — {type(exc).__name__}: {exc}', flush=True)
    print(f'Completed {len(bulk_rows)}/40. Failures: {len(bulk_errors)}')
    if bulk_errors:
        display(pd.DataFrame(bulk_errors))


## Phase 3 — Export validated environment tables and manifest

This table is the independent environmental modality dataset for Notebook 06. The nine values remain **raw derived descriptors**; `Dense(32) → Dense(16)` is learned later during model training, using only training-split scaling parameters. Any source failures must be investigated rather than filled with fabricated values.

In [ ]:
# Cell 10 — Save one row per catchment, no fabricated values or duplicate output files.
# Changing any source/config will alter the fingerprint, invalidating stale caches.
def write_text_if_changed(path, content):
    path = Path(path)
    if path.is_file() and path.read_text(encoding='utf-8') == content:
        return 'UNCHANGED'
    tmp = path.with_name(path.name+'.writing.tmp')
    tmp.write_text(content, encoding='utf-8')
    os.replace(tmp, path)
    return 'WRITTEN'

if not RUN_ALL_40:
    print('Final export skipped: RUN_ALL_40=False. The pilot partial cache remains available.')
elif bulk_errors or len(bulk_rows) != 40:
    print('Final export stopped: process failed stations first. Cached successes remain valid.')
else:
    feature_rows = [{'station_id': r['station_id'], **r['features']} for r in bulk_rows]
    qa_rows = [{'station_id': r['station_id'], 'basin': r['basin'],
                'polygon_repo_path': r['polygon_repo_path'],
                'polygon_sha256': r['polygon_sha256'],
                **r['qa']} for r in bulk_rows]
    feature_table = pd.DataFrame(feature_rows).sort_values('station_id').reset_index(drop=True)
    qa_table = pd.DataFrame(qa_rows).sort_values('station_id').reset_index(drop=True)
    assert feature_table['station_id'].is_unique and len(feature_table) == 40
    assert feature_table[FEATURE_COLS].apply(pd.to_numeric).notna().all().all()
    for field in FEATURE_COLS:
        assert np.isfinite(feature_table[field].astype(float)).all(), field
    manifest = {
        'feature_version': SETTINGS['feature_version'],
        'station_count': 40,
        'feature_order': FEATURE_COLS,
        'source_config': SETTINGS,
        'jrc_zero_policy': 'Never-detected/masked transition areas treated as zero persistent water, not as an observation coverage estimate.',
        'station_polygon_sha256': {r['station_id']: r['polygon_sha256'] for r in bulk_rows},
        'source_fingerprints': {r['station_id']: r['fingerprint'] for r in bulk_rows},
    }
    destinations = [
        (OUT_DIR/'c4_environment_features_40_PRIVATE.csv', feature_table.to_csv(index=False, float_format='%.10g')),
        (OUT_DIR/'c4_environment_QA_40_PRIVATE.csv', qa_table.to_csv(index=False, float_format='%.10g')),
        (OUT_DIR/'c4_environment_manifest_PRIVATE.json', json.dumps(manifest,indent=2,sort_keys=True,allow_nan=False)+'\n'),
    ]
    for path, content in destinations:
        print(write_text_if_changed(path, content), path)
    print('PASS: 40 environment feature vectors saved. Notebook 06 can join on station_id.')
    display(feature_table.head())

In [18]:

# Cell 11 — Verify JRC transition classes vs seasonality
# Read-only QA; does not change the 40-station feature table.

jrc_image = ee.Image(JRC_ASSET)
transition = jrc_image.select("transition")
seasonality = jrc_image.select("seasonality")

persistent = (
    transition.eq(1)
    .Or(transition.eq(2))
    .Or(transition.eq(7))
    .unmask(0)
)

season12 = seasonality.eq(12).unmask(0)

# Pixels on which these two classifications disagree.
disagreement = persistent.neq(season12)

stations_to_check = [
    "kithulgala",
    "thambuththegama",
    "badalgama",
]

for sid in stations_to_check:
    region = gee_polygon(polygons[sid])

    check_image = (
        ee.Image.pixelArea()
        .multiply(disagreement)
        .rename("disagreement_m2")
    )

    stats = ee_area_stats(check_image, region, JRC_SCALE_M)

    disagreement_km2 = (
        float(stats["disagreement_m2"]) / 1e6
    )

    print(
        f"{sid}: JRC mask disagreement = "
        f"{disagreement_km2:.6f} km²"
    )

kithulgala: JRC mask disagreement = 0.000000 km²
thambuththegama: JRC mask disagreement = 0.000000 km²
badalgama: JRC mask disagreement = 0.000000 km²
